In [119]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import time

In [53]:
driver = webdriver.Chrome()
driver.maximize_window()

#Explicit waits
wait = WebDriverWait(driver, 10)

#Function to check site loading
def wait_for_page_to_laod(driver, wait):
    page_title = driver.title
    try:
        wait.until(
            lambda d: d.execute_scrpit('return document.readyState') == 'complete'
            )
    except:
        print(f'The page {page_title} is not get fully loaded!')
    else:
        print(f'The page {page_title} is fully loaded!')


url = 'https://finance.yahoo.com/'
driver.get(url)
wait_for_page_to_laod(driver, wait)

# Hovering on market menu
actions = ActionChains(driver)
markets_menu = wait.until(
    EC.presence_of_element_located((By.XPATH, '/html[1]/body[1]/div[2]/header[1]/div[1]/nav[1]/ol[1]/li[3]/a[1]/div[1]'))
)
actions.move_to_element(markets_menu).perform()

#Hovering over stocks
actions = ActionChains(driver)
stocks_menu = wait.until(
    EC.element_to_be_clickable((By.XPATH, '/html[1]/body[1]/div[2]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/a[1]'))
)
stocks_menu.click()


#Scraping the data

data = []
while True:
    #Scraping
    wait.until(
        EC.presence_of_element_located((By.TAG_NAME, 'table'))
    )
    rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
    for row in rows:
        values = row.find_elements(By.TAG_NAME, 'td')
        
        stocks = {
            'name': values[1].text,
            'symbol':values[0].text,
            'price':values[3].text,
            'change':values[4].text,
            'volume':values[6].text,
            'market_cap':values[8].text,
            'PE_ratio':values[9].text
        }
        data.append(stocks)
        
    
    #Click next
    
    try:
        next_button = wait.until(
            EC.element_to_be_clickable((By.XPATH, '//*[@id="main-content-wrapper"]/section[1]/div/div[4]/div[3]/button[3]'))
        )
    except:
        print("The next button can not be clickable. We have navigated through all the pages")
        break
    else:
        next_button.click()
        time.sleep(2)


The page Yahoo Finance - Stock Market Live, Quotes, Business & Finance News is not get fully loaded!
The next button can not be clickable. We have navigated through all the pages


In [55]:
data

[{'name': 'Warner Bros. Discovery, Inc.',
  'symbol': 'WBD',
  'price': '26.08',
  'change': '+1.54',
  'volume': '189.278M',
  'market_cap': '64.626B',
  'PE_ratio': '126.57'},
 {'name': 'SoFi Technologies, Inc.',
  'symbol': 'SOFI',
  'price': '27.78',
  'change': '-1.82',
  'volume': '129.63M',
  'market_cap': '33.5B',
  'PE_ratio': '53.62'},
 {'name': 'Netflix, Inc.',
  'symbol': 'NFLX',
  'price': '100.24',
  'change': '-2.98',
  'volume': '125.152M',
  'market_cap': '424.749B',
  'PE_ratio': '43.45'},
 {'name': 'NVIDIA Corporation',
  'symbol': 'NVDA',
  'price': '182.41',
  'change': '-0.97',
  'volume': '121.464M',
  'market_cap': '4.441T',
  'PE_ratio': '44.45'},
 {'name': 'HanesBrands Inc.',
  'symbol': 'HBI',
  'price': '6.47',
  'change': '0.00',
  'volume': '112.068M',
  'market_cap': '2.289B',
  'PE_ratio': '5.52'},
 {'name': 'BigBear.ai Holdings, Inc.',
  'symbol': 'BBAI',
  'price': '6.82',
  'change': '-0.20',
  'volume': '110.429M',
  'market_cap': '2.977B',
  'PE_rat

In [57]:
len(data)

288

In [151]:
stocks_data = (
    pd
    .DataFrame(data)
    .apply(lambda col: col.str.strip() if col.dtype=='object' else col)
    .assign(
        price=lambda df_:pd.to_numeric(df_.price),
        change=lambda df_:pd.to_numeric(df_.change.str.replace('+', '')),
        volume=lambda df_:pd.to_numeric(df_.volume.str.replace('M', '')),
        market_cap=lambda df_:df_.market_cap.apply(lambda val:float(val.replace('B', '')) if 'B' in val else float(val.replace('T', ''))*1000),
        PE_ratio=lambda df_:(
            df_
            .PE_ratio
            .replace("-", np.nan))
            .str.replace(",", '')
    )
    .rename(columns={
        'price':'price_usd',
        'volume':'volume_M',
        'market_cap':'market_cap_B'
    })
    


    
)
stocks_data

name             object
symbol           object
price_usd       float64
change          float64
volume_M        float64
market_cap_B    float64
PE_ratio         object
dtype: object

In [153]:
stocks_data.to_excel('yahoo-stocks-data.xlsx', index=False)